In [1]:
top_dir =  '/Users/jcasalet/Desktop/RESEARCH/LIVER/DATA/HI/4_CRISP/'
output_dir = paste(top_dir, 'MERGE_THEN_NORMALIZE', sep="/")

bob = read.csv(paste(top_dir, 'bob.csv', sep=""), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)

dim(bob)



[1] 2 0

## read in expr data

In [2]:
glds_47_complete <- read.csv(paste(top_dir, 'glds_47_complete.csv', sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)
glds_48_complete <- read.csv(paste(top_dir, 'glds_48_complete.csv', sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)
glds_137_complete <- read.csv(paste(top_dir, 'glds_137_complete.csv',sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)
glds_168_complete <- read.csv(paste(top_dir, 'glds_168_complete.csv',sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)

glds_47_no_pseudogenes <- read.csv(paste(top_dir, 'glds_47_no_pseudogenes.csv', sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)
glds_48_no_pseudogenes <- read.csv(paste(top_dir, 'glds_48_no_pseudogenes.csv', sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)
glds_137_no_pseudogenes <- read.csv(paste(top_dir, 'glds_137_no_pseudogenes.csv',sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)
glds_168_no_pseudogenes <- read.csv(paste(top_dir, 'glds_168_no_pseudogenes.csv',sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)

glds_47_no_pseudogenes_wExternal <- read.csv(paste(top_dir, 'glds_47_no_pseudogenes_wExternal.csv', sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)
glds_48_no_pseudogenes_wExternal <- read.csv(paste(top_dir, 'glds_48_no_pseudogenes_wExternal.csv', sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)
glds_137_no_pseudogenes_wExternal <- read.csv(paste(top_dir, 'glds_137_no_pseudogenes_wExternal.csv',sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)
glds_168_no_pseudogenes_wExternal <- read.csv(paste(top_dir, 'glds_168_no_pseudogenes_wExternal.csv',sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)

## Merge unnormalized dataframes 

merge() with by=0 merges dataframes on row names 

merge() with all=FALSE removes the merging elements that do not match between dataframes

In [3]:
# Function to merge 2 dfs on row names and remove non-common row names
uniqMerge <- function(df1, df2){
    merged <- merge(df1, df2, by=0, all=FALSE, no.dups=TRUE) # do the merge
    row.names(merged) = merged$Row.names # assign row names using the "Row.names" column
    merged <- within(merged, rm('Row.names')) # remove the "Row.names" column
}

In [4]:
merged_complete_1 <- uniqMerge(glds_47_complete, glds_48_complete)
merged_complete_2 <- uniqMerge(glds_137_complete, glds_168_complete)
merged_complete <- uniqMerge(merged_complete_1, merged_complete_2)
dim(merged_complete)
merged_no_pseudogenes_1 <- uniqMerge(glds_47_no_pseudogenes, glds_48_no_pseudogenes)
merged_no_pseudogenes_2 <- uniqMerge(glds_137_no_pseudogenes, glds_168_no_pseudogenes)
merged_no_pseudogenes <- uniqMerge(merged_no_pseudogenes_1, merged_no_pseudogenes_2)
dim(merged_no_pseudogenes)
merged_no_pseudogenes_wExternal_1 <- uniqMerge(glds_47_no_pseudogenes_wExternal, glds_48_no_pseudogenes_wExternal)
merged_no_pseudogenes_wExternal_2 <- uniqMerge(glds_137_no_pseudogenes_wExternal, glds_168_no_pseudogenes_wExternal)
merged_no_pseudogenes_wExternal <- uniqMerge(merged_no_pseudogenes_wExternal_1, merged_no_pseudogenes_wExternal_2)
dim(merged_no_pseudogenes_wExternal)

[1] 55536    51

[1] 41983    51

[1] 38500    51

## Metadata (required to make dds object but not used for normalization) Now, used to subset the expression data to samples that have ORO values

In [5]:

meta <- read.csv(paste(top_dir, 'metadata.csv', sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE)
length(rownames(meta))

[1] 51

In [6]:
# subset to only samples with ORO values
merged_complete_final <- merged_complete[rownames(meta)]
merged_no_pseudogenes_final <- merged_no_pseudogenes[rownames(meta)]
merged_no_pseudogenes_wExternal_final <- merged_no_pseudogenes_wExternal[rownames(meta)]
length(merged_complete_final)
length(merged_no_pseudogenes_final)
length(merged_no_pseudogenes_final)

[1] 51

[1] 51

[1] 51

## write unormalized merged data to CSV 

In [7]:
write.csv(merged_complete_final, paste(top_dir,'merged_unnormalized_complete.csv', sep="/"), row.names=TRUE, quote=FALSE)
write.csv(merged_no_pseudogenes_final, paste(top_dir,'merged_unnormalized_no_pseudogenes.csv', sep="/"), row.names=TRUE, quote=FALSE)
write.csv(merged_no_pseudogenes_wExternal_final, paste(top_dir,'merged_unnormalized_no_pseudogenes_wExternal.csv', sep="/"), row.names=TRUE, quote=FALSE)

## Create DESeq Dataset Object

In [8]:
library(DESeq2)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, setdiff, sort, table, tapply,
    union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loading required package: SummarizedExperiment

Loading required package: MatrixGe

In [9]:
# Make dds object after converting values to integers with ceiling()
dds_complete <- DESeqDataSetFromMatrix(ceiling(merged_complete_final), meta['dataset'], ~1)
summary(dds_complete)

dds_no_pseudogenes <- DESeqDataSetFromMatrix(ceiling(merged_no_pseudogenes_final), meta['dataset'], ~1)
summary(dds_complete)

dds_no_pseudogenes_wExternal <- DESeqDataSetFromMatrix(ceiling(merged_no_pseudogenes_wExternal_final), meta['dataset'], ~1)
summary(dds_complete)

converting counts to integer mode



[1] "DESeqDataSet object of length 55536 with 0 metadata columns"

converting counts to integer mode



[1] "DESeqDataSet object of length 55536 with 0 metadata columns"

converting counts to integer mode



[1] "DESeqDataSet object of length 55536 with 0 metadata columns"

## Depth normalization 
Normalize across samples for sequencing depth using median of ratios method. 

Detailed explanation of this method: https://hbctraining.github.io/DGE_workshop/lessons/02_DGE_count_normalization.html

In [10]:
dds_complete <- estimateSizeFactors(dds_complete)
dds_no_pseudogenes <- estimateSizeFactors(dds_no_pseudogenes)
dds_no_pseudogenes_wExternal <- estimateSizeFactors(dds_no_pseudogenes_wExternal)

In [11]:
norm_complete <- counts(dds_complete, normalized=TRUE)
unnorm_complete <- counts(dds_complete, normalized=FALSE)

norm_no_pseudogenes <- counts(dds_no_pseudogenes, normalized=TRUE)
unnorm_no_pseudogenes<- counts(dds_no_pseudogenes, normalized=FALSE)

norm_no_pseudogenes_wExternal <- counts(dds_no_pseudogenes_wExternal, normalized=TRUE)
unnorm_no_pseudogenes_wExternal <- counts(dds_no_pseudogenes_wExternal, normalized=FALSE)

## Batch effect correction 
Since the library prep batch effect is still fairly pronounced after sequencing depth correction, we can correct the data using Combat-Seq (paper: https://pubmed.ncbi.nlm.nih.gov/33015620/)

In [12]:
library(sva)

Loading required package: mgcv

Loading required package: nlme


Attaching package: ‘nlme’


The following object is masked from ‘package:IRanges’:

    collapse


This is mgcv 1.8-39. For overview type 'help("mgcv-package")'.

Loading required package: genefilter


Attaching package: ‘genefilter’


The following objects are masked from ‘package:MatrixGenerics’:

    rowSds, rowVars


The following objects are masked from ‘package:matrixStats’:

    rowSds, rowVars


Loading required package: BiocParallel



In [13]:
norm_complete_Combat <- ComBat_seq(norm_complete, batch=meta$Library.prep)
norm_no_pseudogenes_Combat <- ComBat_seq(norm_no_pseudogenes, batch=meta$Library.prep)
norm_no_pseudogenes_wExternal_Combat <- ComBat_seq(norm_no_pseudogenes_wExternal, batch=meta$Library.prep)


Found 2 batches
Using null model in ComBat-seq.
Adjusting for 0 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data
Found 2 batches
Using null model in ComBat-seq.
Adjusting for 0 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data
Found 2 batches
Using null model in ComBat-seq.
Adjusting for 0 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data


## Convert to gene symbols
Map ENSEMBL gene IDs to gene symbols to make the results easier to interpret biologically. Write out these data.

In [14]:
library(STRINGdb)
library(org.Mm.eg.db)

Loading required package: AnnotationDbi





In [15]:
# get ENSEMBL:symbol mapping from org.Mm.eg.db database
# drop any genes that don't have a gene symbol
mapped <- na.omit(as.data.frame(mapIds(org.Mm.eg.db, keys=rownames(unnorm_complete),
                         keytype='ENSEMBL', column='SYMBOL', multiVals='first')))
colnames(mapped) <- 'symbol'

'select()' returned 1:many mapping between keys and columns



In [16]:
# # Convert and write out unnormalized data
symbolize_and_save <- function(df, fileName){
    temp <- merge(df, mapped, by=0, all=FALSE, no.dups=FALSE) # merge gene symbols into expression df
    .rowNamesDF(temp, make.names=TRUE) <- temp$symbol # make gene symbols row names
    temp <- within(temp, rm(Row.names)) # remove residual columns
    temp <- within(temp, rm(symbol))
    temp <- log2(temp+1)

    write.csv(temp, fileName, row.names=TRUE, quote=FALSE) # write out
}

In [17]:
symbolize_and_save(unnorm_complete, paste(output_dir, 'merged_unnormalized_log_complete.csv', sep="/"))
symbolize_and_save(norm_complete, paste(output_dir, 'merged_normalized_log_complete.csv', sep="/"))
symbolize_and_save(norm_complete_Combat, paste(output_dir, 'merged_normalized_corrected_log_complete.csv', sep="/"))

symbolize_and_save(unnorm_no_pseudogenes, paste(output_dir, 'merged_unnormalized_log_no_pseudogenes.csv', sep="/"))
symbolize_and_save(norm_no_pseudogenes, paste(output_dir, 'merged_normalized_log_no_pseudogenes.csv', sep="/"))
symbolize_and_save(norm_no_pseudogenes_Combat, paste(output_dir, 'merged_normalized_corrected_log_no_pseudogenes.csv', sep="/"))

symbolize_and_save(unnorm_no_pseudogenes_wExternal, paste(output_dir, 'merged_unnormalized_log_no_pseudogenes_wExternal.csv', sep="/"))
symbolize_and_save(norm_no_pseudogenes_wExternal, paste(output_dir, 'merged_normalized_log_no_pseudogenes_wExternal.csv', sep="/"))
symbolize_and_save(norm_no_pseudogenes_wExternal_Combat, paste(output_dir, 'merged_normalized_corrected_log_no_pseudogenes_wExternal.csv', sep="/"))